Models can request to call tools that perform tasks such as fetching data from database, searching the web, or running code. Tools are pairings of:

    1. a schema, including the name of the tool, a descriptions, and/or argument definations(often a JSON schema)
    2.A function or coroutine to exxcute.

In [21]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

res = model.invoke("Write an essay on Gandhiji in South africa")

res

AIMessage(content="<think>\nOkay, the user wants an essay on Gandhiji in South Africa. Let me start by recalling what I know. Gandhi spent 21 years in South Africa, right? He went there in 1893 as a lawyer. The initial incident where he was thrown off the train in Pietermaritzburg is a key point. That event probably sparked his activism.\n\nI should mention his key contributions there. Satyagraha comes to mind. He developed this philosophy of non-violent resistance. The Indian community in South Africa faced discrimination, so Gandhi organized protests against laws like the Asiatic Registration Act. Maybe include the Bloemfontein incident where he led a campaign to repeal the law.\n\nAlso, the Transvaal Passive Resistance Association. He emphasized self-reliance, like spinning khadi. His experiments with community living, like Tolstoy Farm. How his time in South Africa shaped his methods and beliefs before he returned to India.\n\nNeed to structure the essay with an introduction, body 

In [ ]:
## Tools
from langchain.tools import tool

@tool
def get_weather(loc:str)->str:
    """Get The Weather info at Location"""
    return f"It's raining heavily with thunderstorms and strong winds in {loc}"

model_with_tools=model.bind_tools([get_weather])




In [23]:
resTv=model_with_tools.invoke("what is the weather in  surat now ?")

print(resTv)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the current weather in Surat. I need to use the get_weather function. The function requires the location parameter, which in this case is "surat". I should structure the tool call with the name as get_weather and the arguments including "loc" set to "surat". Let me make sure the JSON is correctly formatted.\n', 'tool_calls': [{'id': 'tn1ense44', 'function': {'arguments': '{"loc":"surat"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 156, 'total_tokens': 255, 'completion_time': 0.156283924, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.008281672, 'prompt_tokens_details': None, 'queue_time': 0.510056778, 'total_time': 0.164565596}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} i

In [24]:
resTv.tool_calls


[{'name': 'get_weather',
  'args': {'loc': 'surat'},
  'id': 'tn1ense44',
  'type': 'tool_call'}]

Tool Execution Loop

In [ ]:
# STEP -1 MODEL GENERATE TOOL CALL
message=[{"role":"user", "content": "What's the weather in Surat now, shold i go out of home?"}]
ai_msg=model_with_tools.invoke(message)
message.append(ai_msg)

# STEP-2 EXXCUTE TOOLS AND COLLECT RESULTS
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the genrated arguments
    tool_res = get_weather.invoke(tool_call)
    message.append(tool_res)

# step 3 Pass results back to model for final res
final_res = model_with_tools.invoke(message)
print(final_res.text)

It's currently raining heavily with thunderstorms and strong winds in Surat. I recommend staying indoors to avoid the risks of flooding, lightning, and wind-related hazards. If you must go out, ensure you have proper protection and avoid flooded areas or unstable structures.


In [26]:
message

[{'role': 'user',
  'content': "What's the weather in Surat now, shold i go out of home?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Surat and whether they should go out. First, I need to check the current weather there. The available tool is get_weather, which requires the location parameter. Surat is a city in India, so I\'ll use that as the location. I\'ll call the get_weather function with loc set to "Surat". Once I get the weather data, I can assess conditions like temperature, precipitation, and wind speed to advise them. If it\'s raining or very hot, maybe they should stay in. If it\'s clear and mild, going out might be fine. I need to make sure the response is clear and helpful based on the actual weather info retrieved.\n', 'tool_calls': [{'id': 'j6gps023n', 'function': {'arguments': '{"loc":"Surat"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 169, 